# 02 — LATCH-Style Tract Disaggregation

**Method:** Adapted from BTS LATCH 2017 (Local Area Transportation Characteristics for Households)  
**Source:** NHTS 2022 + ACS 2022 5-year tract estimates  
**Geography:** Census tract (~84k tracts, all 50 states + DC)

## Methodology overview

The NHTS 2022 provides household travel behavior at national scale but lacks sub-state geographic identifiers. This notebook disaggregates NHTS estimates to Census tract level by combining two approaches:

**1. Weighted least squares regression — for total travel volumes**  
Fit separately for each of 18 strata (6 Census division groups × 3 urban types). Each model predicts daily household travel (VMT, person miles, vehicle trips, person trips) as a function of household demographics. Regression is weighted by NHTS household weights (WTHHFIN). Coefficients are then applied to ACS tract-level demographics to produce tract estimates.  
→ 18 strata keeps cell sizes robust given the NHTS 2022 sample (~7,800 households).

**2. Direct stratum shares — for purpose and mode breakdown**  
Weighted shares of trips by purpose (WHYTO) and mode (TRPTRANS) are computed directly from the NHTS trip file within each stratum. No regression needed — shares are applied as post-stratification rates on top of the tract-level totals.  
→ 27 strata (9 Census divisions × 3 urban types) preserves full geographic granularity for behavioral breakdown.

**Urban classification** uses the same TIGER 2020 Urban Area polygons that NHTS used to derive the `URBAN` variable — a direct methodological match.

**Strata summary:**
| Use | Strata | Definition |
|---|---|---|
| Regression | 18 | 6 division groups × 3 URBAN types |
| Shares | 27 | 9 Census divisions × 3 URBAN types |

**Outputs per tract:**
- `vmt_per_hh`, `person_miles_per_hh`, `veh_trips_per_hh`, `person_trips_per_hh` — regression estimates
- `total_*` versions scaled by number of households (ACS B11001)
- `trips_whyto_<code>` — person trips by trip purpose
- `trips_trptrans_<code>` — person trips by mode

**Reference:** [BTS LATCH 2017 Methodology](https://www.bts.gov/latch/latch-methodology-2017)

## 0. Imports and configuration

In [17]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import requests
from sklearn.linear_model import LinearRegression
import pygris
import warnings
warnings.filterwarnings("ignore")

In [18]:
DATA_DIR   = Path(r"C:\Users\m.kroetz\OneDrive - Systematica\HONDA\repo\data")
HH_CSV     = DATA_DIR / "NHTS" / "csv" / "hhv2pub.csv"
PER_CSV    = DATA_DIR / "NHTS" / "csv" / "perv2pub.csv"
TRIP_CSV   = DATA_DIR / "NHTS" / "csv" / "tripv2pub.csv"
OUTPUT_DIR = Path(r"C:\Users\m.kroetz\OneDrive - Systematica\HONDA\repo\outputs\latch")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CENSUS_API_KEY = "fb4d52bc047ea2219e5e7f68ce7ca7b7cdcae73e"

ALL_STATE_FIPS = [
    "01","02","04","05","06","08","09","10","11","12","13","15","16","17","18",
    "19","20","21","22","23","24","25","26","27","28","29","30","31","32","33",
    "34","35","36","37","38","39","40","41","42","44","45","46","47","48","49",
    "50","51","53","54","55","56"
]

# 9 Census divisions → 6 region groups (matching 2017 LATCH)
# Groups: 1=Northeast, 2=Midwest, 3=South Atlantic, 4=South Central, 5=Mountain, 6=Pacific
DIVISION_TO_GROUP = {
    1: 1, 2: 1,   # Northeast (New England + Middle Atlantic)
    3: 2, 4: 2,   # Midwest (East + West North Central)
    5: 3,         # South Atlantic
    6: 4, 7: 4,   # South Central (East + West combined)
    8: 5,         # Mountain
    9: 6,         # Pacific
}
GROUP_NAMES = {
    1: "Northeast", 2: "Midwest", 3: "South Atlantic",
    4: "South Central", 5: "Mountain", 6: "Pacific"
}

# State FIPS → Census Division (for tract stratum assignment)
STATE_TO_DIVISION = {
    "09":1,"23":1,"25":1,"33":1,"44":1,"50":1,  # New England
    "34":2,"36":2,"42":2,                          # Middle Atlantic
    "17":3,"18":3,"26":3,"39":3,"55":3,           # East North Central
    "19":4,"20":4,"27":4,"29":4,"31":4,"38":4,"46":4,  # West North Central
    "10":5,"11":5,"12":5,"13":5,"24":5,"37":5,"45":5,"51":5,"54":5,  # South Atlantic
    "01":6,"21":6,"28":6,"47":6,                   # East South Central
    "05":7,"22":7,"40":7,"48":7,                   # West South Central
    "04":8,"08":8,"16":8,"30":8,"32":8,"35":8,"49":8,"56":8,  # Mountain
    "02":9,"06":9,"15":9,"41":9,"53":9,           # Pacific
}

## 1. Load and prepare NHTS data

Merges household, person, and trip files.

**Life cycle dummies** (from 2017 LATCH spec, derived from person file):
- `lc_children`: household has at least one child aged 5–17
- `lc_1adult_u65`: single-adult household, head under 65
- `lc_2plus_no65`: 2+ adult household, no one aged 65+
- `lc_2plus_65`: 2+ adult household, at least one person 65+ (reference category = 1 adult aged 65+)

**Vehicle and worker dummies** replace counts (reference = 0 vehicles / 0 workers).

In [19]:
hh = pd.read_csv(HH_CSV, usecols=[
    "HOUSEID", "WTHHFIN", "CENSUS_D", "URBAN",
    "HHVEHCNT", "HHSIZE", "WRKCOUNT", "HHFAMINC"
], low_memory=False)

per = pd.read_csv(PER_CSV, usecols=["HOUSEID", "R_AGE"], low_memory=False)

trip = pd.read_csv(TRIP_CSV, usecols=[
    "HOUSEID", "TRPMILES", "WTTRDFIN", "WHYTO", "TRPTRANS", "VEHID"
], low_memory=False)

# Valid trips
trip = trip[(trip["TRPMILES"] > 0) & (trip["WTTRDFIN"] > 0) & (trip["WHYTO"] > 0)].copy()
trip["WHYTO"]    = trip["WHYTO"].astype(int)
trip["TRPTRANS"] = trip["TRPTRANS"].astype(int)

# Household-level travel day totals (unweighted — regression weighted by WTHHFIN)
hh_trip = trip.groupby("HOUSEID").agg(
    hh_person_miles  = ("TRPMILES", "sum"),
    hh_person_trips  = ("HOUSEID",  "count"),
).reset_index()
# Vehicle trips = trips made in a private vehicle (VEHID > 0)
veh_trips = trip[trip["VEHID"] > 0].groupby("HOUSEID").agg(
    hh_veh_trips = ("HOUSEID", "count")
).reset_index()
# VMT = miles in private vehicle trips only
vmt = trip[trip["VEHID"] > 0].groupby("HOUSEID").agg(
    hh_vmt = ("TRPMILES", "sum")
).reset_index()

hh = hh.merge(hh_trip, on="HOUSEID", how="left")
hh = hh.merge(veh_trips, on="HOUSEID", how="left")
hh = hh.merge(vmt, on="HOUSEID", how="left")
hh[["hh_person_miles","hh_person_trips","hh_veh_trips","hh_vmt"]] = \
    hh[["hh_person_miles","hh_person_trips","hh_veh_trips","hh_vmt"]].fillna(0)

# Income midpoints (NHTS 2022 HHFAMINC codes)
INCOME_MAP = {
    1:5_000, 2:12_500, 3:20_000, 4:30_000, 5:42_500,
    6:62_500, 7:87_500, 8:112_500, 9:137_500, 10:175_000, 11:225_000
}
hh["income"] = hh["HHFAMINC"].map(INCOME_MAP)

# Vehicle dummies (reference: 0 vehicles)
hh["veh_1"]    = (hh["HHVEHCNT"] == 1).astype(int)
hh["veh_2plus"] = (hh["HHVEHCNT"] >= 2).astype(int)

# Worker dummies (reference: 0 workers)
hh["wrk_1"]    = (hh["WRKCOUNT"] == 1).astype(int)
hh["wrk_2plus"] = (hh["WRKCOUNT"] >= 2).astype(int)

# Life cycle dummies from person file
per["R_AGE"] = pd.to_numeric(per["R_AGE"], errors="coerce")
per_valid = per[per["R_AGE"] >= 0]

has_children = per_valid[per_valid["R_AGE"].between(5, 17)].groupby("HOUSEID").size().gt(0).rename("lc_children")
age_max      = per_valid.groupby("HOUSEID")["R_AGE"].max().rename("max_age")
age_min      = per_valid.groupby("HOUSEID")["R_AGE"].min().rename("min_age")
hh_size_per  = per_valid.groupby("HOUSEID").size().rename("n_persons")

lc = pd.concat([has_children, age_max, age_min, hh_size_per], axis=1).reset_index()
lc["lc_children"]  = lc["lc_children"].fillna(False).astype(int)
lc["lc_1adult_u65"] = ((lc["n_persons"] == 1) & (lc["max_age"] < 65)).astype(int)
lc["lc_2plus_no65"] = ((lc["n_persons"] >= 2) & (lc["max_age"] < 65)).astype(int)
lc["lc_2plus_65"]   = ((lc["n_persons"] >= 2) & (lc["max_age"] >= 65)).astype(int)
# Reference category = 1 adult aged 65+ alone

hh = hh.merge(lc[["HOUSEID","lc_children","lc_1adult_u65","lc_2plus_no65","lc_2plus_65"]], on="HOUSEID", how="left")

# Drop rows missing required fields
required = ["WTHHFIN","CENSUS_D","URBAN","income",
            "veh_1","veh_2plus","wrk_1","wrk_2plus",
            "lc_children","lc_1adult_u65","lc_2plus_no65","lc_2plus_65"]
hh = hh.dropna(subset=required)
hh["CENSUS_D"] = hh["CENSUS_D"].astype(int)
hh["URBAN"]    = hh["URBAN"].astype(int)
hh = hh[hh["URBAN"].isin([1, 2, 4])].copy()

# Strata
hh["div_group"] = hh["CENSUS_D"].map(DIVISION_TO_GROUP)  # 6 groups — for regression
hh["stratum_reg"] = hh["div_group"].astype(str) + "_" + hh["URBAN"].astype(str)   # 18 strata
hh["stratum_shr"] = hh["CENSUS_D"].astype(str) + "_" + hh["URBAN"].astype(str)   # 27 strata

print(f"Households: {len(hh):,}")
print("\nRegression strata (18):")
print(hh.groupby("stratum_reg")["HOUSEID"].count().sort_index())

Households: 7,796

Regression strata (18):
stratum_reg
1_1    1021
1_2     104
1_4     218
2_1    1136
2_2     184
2_4     515
3_1    1163
3_2      92
3_4     305
4_1     859
4_2     138
4_4     325
5_1     441
5_2      72
5_4     100
6_1     953
6_2      77
6_4      93
Name: HOUSEID, dtype: int64


## 2. WLS regression per stratum

Four dependent variables (matching 2017 LATCH): VMT, person miles, vehicle trips, person trips.  
Nine predictors: income, 2 vehicle dummies, 2 worker dummies, 4 life cycle dummies.  
Fitted on 18 strata using NHTS household weights (WTHHFIN).

In [20]:
PREDICTORS = ["income","veh_1","veh_2plus","wrk_1","wrk_2plus",
              "lc_children","lc_1adult_u65","lc_2plus_no65","lc_2plus_65"]
TARGETS    = ["hh_vmt","hh_person_miles","hh_veh_trips","hh_person_trips"]

stratum_models = {}
for stratum, group in hh.groupby("stratum_reg"):
    X = group[PREDICTORS].values
    w = group["WTHHFIN"].values
    stratum_models[stratum] = {
        t: LinearRegression().fit(X, group[t].values, sample_weight=w)
        for t in TARGETS
    }

print(f"Fitted models for {len(stratum_models)} strata")

# R² diagnostic
rows = []
for stratum, group in hh.groupby("stratum_reg"):
    X = group[PREDICTORS].values
    w = group["WTHHFIN"].values
    for t in TARGETS:
        r2 = stratum_models[stratum][t].score(X, group[t].values, sample_weight=w)
        rows.append({"stratum": stratum, "target": t, "n": len(group), "r2": round(r2, 3)})

pd.DataFrame(rows).sort_values(["target","stratum"])

Fitted models for 18 strata


,stratum,target,n,r2
1,1_1,hh_person_miles,1021,0.050
5,1_2,hh_person_miles,104,0.151
9,1_4,hh_person_miles,218,0.057
13,2_1,hh_person_miles,1136,0.049
17,2_2,hh_person_miles,184,0.152
...,...,...,...,...
52,5_2,hh_vmt,72,0.162
56,5_4,hh_vmt,100,0.087
60,6_1,hh_vmt,953,0.061
64,6_2,hh_vmt,77,0.130


## 3. Purpose and mode shares per stratum

Weighted shares computed directly from the trip file — no regression.  
Uses 27 strata (9 Census divisions × 3 URBAN) for maximum geographic granularity.

In [21]:
trip_hh = trip.merge(hh[["HOUSEID","stratum_shr","WTHHFIN"]], on="HOUSEID", how="inner")

def stratum_shares(df, code_col):
    total     = df.groupby("stratum_shr")["WTHHFIN"].sum()
    breakdown = df.groupby(["stratum_shr", code_col])["WTHHFIN"].sum()
    return (breakdown / total).reset_index().rename(columns={"WTHHFIN": "share"})

purpose_shares = stratum_shares(trip_hh, "WHYTO")
mode_shares    = stratum_shares(trip_hh, "TRPTRANS")

purpose_wide = purpose_shares.pivot(index="stratum_shr", columns="WHYTO",    values="share").add_prefix("p_").reset_index()
mode_wide    = mode_shares.pivot(   index="stratum_shr", columns="TRPTRANS", values="share").add_prefix("m_").reset_index()

print(f"Share strata (27): {purpose_wide['stratum_shr'].nunique()}")
print(f"Purpose codes: {sorted([int(c[2:]) for c in purpose_wide.columns if c != 'stratum_shr'])}")
print(f"Mode codes:    {sorted([int(c[2:]) for c in mode_wide.columns    if c != 'stratum_shr'])}")

Share strata (27): 27
Purpose codes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 97]
Mode codes:    [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]


## 4. Pull ACS 2022 tract data

ACS variables used as regression predictors at tract level.  
Vehicle and worker counts are converted to proportions matching the dummy variable structure used in the NHTS regression.

In [22]:
ACS_VARS = {
    "B19013_001E": "median_income",
    "B11001_001E": "total_hh",
    "B01003_001E": "total_pop",
    "B08201_001E": "veh_denom",
    "B08201_003E": "veh_1_hh",
    "B08201_004E": "veh_2_hh",
    "B08201_005E": "veh_3_hh",
    "B08201_006E": "veh_4plus_hh",
    "B08202_001E": "wrk_denom",
    "B08202_003E": "wrk_1_hh",
    "B08202_004E": "wrk_2_hh",
    "B08202_005E": "wrk_3plus_hh",
    # Life cycle proxies
    "B11005_002E": "hh_with_children",   # HHs with children under 18
    "B11005_001E": "hh_total_lc",
    "B09021_002E": "pop_u18",             # Population under 18 (for age structure)
    "B01001_020E": "pop_65plus_m",        # Male 65+
    "B01001_044E": "pop_65plus_f",        # Female 65+
}

base_url = "https://api.census.gov/data/2022/acs/acs5"
var_str  = ",".join(ACS_VARS.keys())

frames = []
for fips in ALL_STATE_FIPS:
    r = requests.get(base_url, params={
        "get": var_str, "for": "tract:*",
        "in": f"state:{fips}", "key": CENSUS_API_KEY
    })
    data = r.json()
    frames.append(pd.DataFrame(data[1:], columns=data[0]))

acs = pd.concat(frames, ignore_index=True).rename(columns=ACS_VARS)
acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]

num_cols = list(ACS_VARS.values())
acs[num_cols] = acs[num_cols].apply(pd.to_numeric, errors="coerce").replace(-666666666, np.nan)

# Proportions matching regression dummy structure
acs["prop_veh_1"]    = acs["veh_1_hh"]  / acs["veh_denom"].replace(0, np.nan)
acs["prop_veh_2plus"] = (acs["veh_2_hh"] + acs["veh_3_hh"] + acs["veh_4plus_hh"]) / acs["veh_denom"].replace(0, np.nan)
acs["prop_wrk_1"]    = acs["wrk_1_hh"]  / acs["wrk_denom"].replace(0, np.nan)
acs["prop_wrk_2plus"] = (acs["wrk_2_hh"] + acs["wrk_3plus_hh"]) / acs["wrk_denom"].replace(0, np.nan)
acs["prop_children"] = acs["hh_with_children"] / acs["hh_total_lc"].replace(0, np.nan)
acs["avg_hh_size"]   = acs["total_pop"] / acs["total_hh"].replace(0, np.nan)
acs["prop_65plus"]   = (acs["pop_65plus_m"] + acs["pop_65plus_f"]) / acs["total_pop"].replace(0, np.nan)

# Life cycle proportions (tract-level approximations)
# lc_1adult_u65: single-person HH, head < 65 — approximate as (1-person HHs) × (1 - prop_65plus)
# lc_2plus_no65: 2+ person, no 65+ — approximate as (multi-person HH share) × (1 - prop_65plus)
# lc_2plus_65:   2+ person, with 65+ — approximate as (multi-person HH share) × prop_65plus
# These are tract-level proportions, applied as continuous predictors
acs["lc_children"]   = acs["prop_children"]
acs["lc_1adult_u65"] = (1 - acs["prop_65plus"]) / acs["avg_hh_size"].replace(0, np.nan)  # proxy
acs["lc_2plus_no65"] = (1 - acs["prop_65plus"]) * (1 - 1/acs["avg_hh_size"].replace(0, np.nan))  # proxy
acs["lc_2plus_65"]   = acs["prop_65plus"]

print(f"Tracts from ACS: {len(acs):,}")

Tracts from ACS: 84,415


## 5. Pull tract geometries

In [23]:
gdfs = [pygris.tracts(state=fips, year=2022, cb=True) for fips in ALL_STATE_FIPS]
tracts_gdf = pd.concat(gdfs, ignore_index=True)
tracts_gdf = tracts_gdf[["GEOID", "geometry"]].to_crs(epsg=5070)

print(f"Tract geometries: {len(tracts_gdf):,}")

Tract geometries: 84,120


## 6. Assign URBAN type to tracts via TIGER 2020 Urban Areas

The NHTS `URBAN` variable was derived by point-in-polygon lookup against TIGER 2020 Urban Area polygons.  
We replicate the same classification for tract centroids:
- Urbanized Area (50k+) → URBAN 1
- Urban Cluster (2,500–50k) → URBAN 2  
- No match → URBAN 4 (rural)

In [25]:
ua = pygris.urban_areas(year=2020)
print(ua.columns.tolist())
print(ua.head(2))

['UACE10', 'GEOID10', 'NAME10', 'NAMELSAD10', 'LSAD10', 'MTFCC10', 'UATYP10', 'FUNCSTAT10', 'ALAND10', 'AWATER10', 'INTPTLAT10', 'INTPTLON10', 'geometry']
  UACE10 GEOID10        NAME10                  NAMELSAD10 LSAD10 MTFCC10  \
0  24310   24310     Dixon, IL     Dixon, IL Urban Cluster     76   G3500   
1  27847   27847  Escanaba, MI  Escanaba, MI Urban Cluster     76   G3500   

  UATYP10 FUNCSTAT10   ALAND10  AWATER10   INTPTLAT10    INTPTLON10  \
0       C          S  25562730    938057  +41.8529507  -089.4817439   
1       C          S  46643391    283456  +45.7274565  -087.0824457   

                                            geometry  
0  POLYGON ((-89.49859 41.85467, -89.49858 41.855...  
1  MULTIPOLYGON (((-87.09518 45.79692, -87.09496 ...  


In [28]:
urban_areas = pygris.urban_areas(year=2020).to_crs(epsg=5070)[["UATYP10", "geometry"]]

tract_centroids = tracts_gdf.copy()
tract_centroids["geometry"] = tracts_gdf.centroid

joined = gpd.sjoin(tract_centroids, urban_areas, how="left", predicate="within")
joined = joined.groupby("GEOID")["UATYP10"].first().reset_index()
joined["URBAN"] = joined["UATYP10"].map({"U": 1, "C": 2}).fillna(4).astype(int)

tracts_gdf = tracts_gdf.merge(joined[["GEOID","URBAN"]], on="GEOID", how="left")
tracts_gdf["URBAN"] = tracts_gdf["URBAN"].fillna(4).astype(int)

print(tracts_gdf["URBAN"].value_counts().sort_index())

URBAN
1    53538
2     4041
4    26541
Name: count, dtype: int64


## 7. Merge ACS data and assign strata

Each tract gets two stratum labels:
- `stratum_reg` — for regression (18 strata: 6 division groups × 3 URBAN)
- `stratum_shr` — for shares (27 strata: 9 Census divisions × 3 URBAN)

In [29]:
tracts = tracts_gdf.merge(acs, on="GEOID", how="inner")
tracts["census_div"]   = tracts["state"].map(STATE_TO_DIVISION)
tracts["div_group"]    = tracts["census_div"].map(DIVISION_TO_GROUP)
tracts["stratum_reg"]  = tracts["div_group"].astype(str)    + "_" + tracts["URBAN"].astype(str)
tracts["stratum_shr"]  = tracts["census_div"].astype(str)   + "_" + tracts["URBAN"].astype(str)

print("Regression strata distribution:")
print(tracts["stratum_reg"].value_counts().sort_index())
print(f"\nTotal tracts: {len(tracts):,}")

Regression strata distribution:
stratum_reg
1_1    10849
1_2      525
1_4     3322
2_1    10621
2_2     1235
2_4     6666
3_1    10456
3_2      528
3_4     5567
4_1     8085
4_2      933
4_4     6591
5_1     3860
5_2      354
5_4     2040
6_1     9667
6_2      466
6_4     2355
Name: count, dtype: int64

Total tracts: 84,120


## 8. Apply regression → tract travel estimates

Regression coefficients from each stratum model are applied to tract-level ACS predictors.  
Any missing ACS values are filled with the stratum mean before prediction.  
Results are clipped at zero (no negative travel estimates).  
Totals are scaled by number of households (ACS B11001).

In [30]:
# ACS predictor order must match NHTS regression predictor order
TRACT_PREDS = ["median_income","prop_veh_1","prop_veh_2plus",
               "prop_wrk_1","prop_wrk_2plus",
               "lc_children","lc_1adult_u65","lc_2plus_no65","lc_2plus_65"]

pred_arrays = {t: np.full(len(tracts), np.nan) for t in TARGETS}

for stratum, idx in tracts.groupby("stratum_reg").groups.items():
    if stratum not in stratum_models:
        continue
    X = tracts.loc[idx, TRACT_PREDS].values
    col_means = np.nanmean(X, axis=0)
    X = np.where(np.isnan(X), col_means, X)
    for t in TARGETS:
        pred_arrays[t][idx] = stratum_models[stratum][t].predict(X).clip(min=0)

for t in TARGETS:
    tracts[t.replace("hh_", "") + "_per_hh"] = pred_arrays[t]
    tracts["total_" + t.replace("hh_", "")] = pred_arrays[t] * tracts["total_hh"]

print(f"Tracts with estimates: {tracts['vmt_per_hh'].notna().sum():,}")
tracts[[c for c in tracts.columns if c.endswith("_per_hh")]].describe().round(2)

Tracts with estimates: 84,120


,vmt_per_hh,person_miles_per_hh,veh_trips_per_hh,person_trips_per_hh
count,84120.00,84120.00,84120.00,84120.00
mean,38.08,51.69,3.23,3.86
std,14.01,19.45,0.87,0.87
min,0.00,0.00,0.00,0.36
25%,28.67,38.93,2.70,3.28
50%,37.52,50.65,3.26,3.82
75%,47.57,62.15,3.76,4.40
max,147.87,259.55,9.93,9.91


## 9. Apply purpose and mode shares

Stratum-level shares (27 strata) are joined to tracts and multiplied by `total_person_trips`  
to produce absolute trip counts by purpose and mode per tract.

In [31]:
tracts = tracts.merge(purpose_wide, on="stratum_shr", how="left")
tracts = tracts.merge(mode_wide,    on="stratum_shr", how="left")

for col in [c for c in tracts.columns if c.startswith("p_")]:
    tracts[f"trips_whyto_{col[2:]}"] = tracts[col] * tracts["total_person_trips"]

for col in [c for c in tracts.columns if c.startswith("m_")]:
    tracts[f"trips_trptrans_{col[2:]}"] = tracts[col] * tracts["total_person_trips"]

tracts = tracts.drop(columns=[c for c in tracts.columns if c.startswith("p_") or c.startswith("m_")])

print("Trip columns:", [c for c in tracts.columns if c.startswith("trips_")])

Trip columns: ['trips_whyto_1', 'trips_whyto_2', 'trips_whyto_3', 'trips_whyto_4', 'trips_whyto_5', 'trips_whyto_6', 'trips_whyto_7', 'trips_whyto_8', 'trips_whyto_9', 'trips_whyto_10', 'trips_whyto_11', 'trips_whyto_12', 'trips_whyto_13', 'trips_whyto_14', 'trips_whyto_15', 'trips_whyto_16', 'trips_whyto_17', 'trips_whyto_18', 'trips_whyto_19', 'trips_whyto_97', 'trips_trptrans_1', 'trips_trptrans_2', 'trips_trptrans_3', 'trips_trptrans_4', 'trips_trptrans_6', 'trips_trptrans_7', 'trips_trptrans_8', 'trips_trptrans_9', 'trips_trptrans_10', 'trips_trptrans_11', 'trips_trptrans_12', 'trips_trptrans_13', 'trips_trptrans_14', 'trips_trptrans_15', 'trips_trptrans_16', 'trips_trptrans_17', 'trips_trptrans_18', 'trips_trptrans_19', 'trips_trptrans_20', 'trips_trptrans_21']


## 10. Export

In [32]:
drop_cols = ["veh_denom","veh_1_hh","veh_2_hh","veh_3_hh","veh_4plus_hh",
             "wrk_denom","wrk_1_hh","wrk_2_hh","wrk_3plus_hh",
             "hh_with_children","hh_total_lc","pop_u18","pop_65plus_m","pop_65plus_f",
             "state","county","tract"]
tracts = tracts.drop(columns=[c for c in drop_cols if c in tracts.columns])

out_gpkg = OUTPUT_DIR / "nhts_latch_tracts.gpkg"
out_csv  = OUTPUT_DIR / "nhts_latch_tracts.csv"

tracts.to_file(out_gpkg, driver="GPKG")
tracts.drop(columns="geometry").to_csv(out_csv, index=False)

print(f"GeoPackage: {out_gpkg}")
print(f"CSV:        {out_csv}")
print(f"Rows: {len(tracts):,} | Columns: {len(tracts.columns)}")

GeoPackage: C:\Users\m.kroetz\OneDrive - Systematica\HONDA\repo\outputs\latch\nhts_latch_tracts.gpkg
CSV:        C:\Users\m.kroetz\OneDrive - Systematica\HONDA\repo\outputs\latch\nhts_latch_tracts.csv
Rows: 84,120 | Columns: 69
